In [38]:
# import functions
import pandas as pd
import numpy as np  
import torch
from torch_geometric.nn import GCNConv, GAE 
from sklearn.preprocessing import StandardScaler

In [39]:
# Load expressions
df_expr = pd.read_csv(f"E:\SPRING26\ECE 542-ML\orthogroup_expression\zma_osa_sbi_ortho_hai_expr_all.tsv", sep='\t')

In [40]:
# Load orthogroups
df_map = pd.read_csv(f"E:\SPRING26\ECE 542-ML\orthogroups\maize_outward\orthos_maize_outward.tsv", sep='\t')

In [41]:
# Load iGRN network
df_net = pd.read_csv(f"E:\SPRING26\ECE 542-ML\Data_file1_iGRN_supervised_network_support.txt", 
                     sep='\t', header=None, usecols=[0, 1])
df_net.columns = ['Source_AT', 'Target_AT']

In [42]:
# This part looks at the column named orthologous_species and checks every single row. 
# If the species is 'ath' (Arabidopsis thaliana), it marks it as True. 
# If it is anything else (like 'aco', 'bdi', or 'osa'), it marks it as False.
# A new table containing ONLY the rows where that condition was True.
df_map_ath = df_map[df_map['orthologous_species']=='ath'].copy()
print(df_map_ath.shape)
print(df_map_ath.head())

(43553, 4)
      #query_gene     OG orthologous_gene orthologous_species
1   GRMZM5G800096  OG001        AT2G07633                 ath
29  GRMZM5G800101  OG002        ATCG01250                 ath
30  GRMZM5G800101  OG002        ATCG00890                 ath
70  GRMZM5G800780  OG003        ATCG00720                 ath
92  GRMZM5G800980  OG004        ATCG00430                 ath


In [43]:
# Extract numeric part from OG (e.g., 'OG001' -> 1)
df_map_ath['numeric_id'] = df_map_ath['OG'].str.extract(r'OG(\d+)').dropna().astype(int)
print(df_map_ath.shape)
print(df_map_ath.head())

(43553, 5)
      #query_gene     OG orthologous_gene orthologous_species  numeric_id
1   GRMZM5G800096  OG001        AT2G07633                 ath           1
29  GRMZM5G800101  OG002        ATCG01250                 ath           2
30  GRMZM5G800101  OG002        ATCG00890                 ath           2
70  GRMZM5G800780  OG003        ATCG00720                 ath           3
92  GRMZM5G800980  OG004        ATCG00430                 ath           4


In [44]:
# Extract numeric part from Expression IDs (e.g., 'ORTHO04M000001' -> 1)
# We fill non-matching IDs (like 'ZeamMp002') with -1 to safely cast to integer
extracted_expr = df_expr['Gene.ID'].str.extract(r'ORTHO04M(\d+)')
df_expr['numeric_id'] = extracted_expr[0].fillna(-1).astype(int)
print(df_expr.shape)
print(df_expr.head())

(18449, 700)
          Gene.ID  mean_leaf.section.1  mean_leaf.section.2  \
0  ORTHO04M000001             4.546497             6.520382   
1  ORTHO04M000002             5.568790            10.509554   
2  ORTHO04M000003             5.755689             4.791018   
3  ORTHO04M000004             5.931737             7.356886   
4  ORTHO04M000005            22.267045            31.086364   

   mean_leaf.section.3  mean_leaf.section.4  mean_leaf.section.6  \
0             6.253503             6.826752             6.745223   
1            11.924204             7.954140             3.438217   
2             4.871257             5.071257             4.532335   
3             9.832934             9.923952             6.170659   
4            33.167045            26.302273            12.117045   

   mean_leaf.section.7  mean_leaf.section.8  mean_leaf.section.9  \
0             3.363057             5.000000             4.141401   
1             0.000000             1.582803             1.53758

In [45]:
# Create the Translation Dictionary
# Link the shared numeric ID to the full Expression ID
valid_expr = df_expr[df_expr['numeric_id'] != -1]
num_to_ortho = dict(zip(valid_expr['numeric_id'], valid_expr['Gene.ID']))
print(f"Translation Dictionary Sample: {list(num_to_ortho.items())[:5]}")

Translation Dictionary Sample: [(1, 'ORTHO04M000001'), (2, 'ORTHO04M000002'), (3, 'ORTHO04M000003'), (4, 'ORTHO04M000004'), (5, 'ORTHO04M000005')]


In [46]:
# Link Arabidopsis locus IDs to the mapped Expression ID
df_map_ath['Target_Ortho'] = df_map_ath['numeric_id'].map(num_to_ortho)
df_map_ath_clean = df_map_ath.dropna(subset=['Target_Ortho'])
at_to_ortho_dict = dict(zip(df_map_ath_clean['orthologous_gene'], df_map_ath_clean['Target_Ortho']))
print(f"Arabidopsis to Expression ID Sample: {list(at_to_ortho_dict.items())[:5]}")

Arabidopsis to Expression ID Sample: [('AT2G07633', 'ORTHO04M000001'), ('ATCG01250', 'ORTHO04M000077'), ('ATCG00890', 'ORTHO04M000077'), ('ATCG00720', 'ORTHO04M000003'), ('ATCG00430', 'ORTHO04M009535')]


In [49]:
#  Map the Network to Ortholog Space
df_net['Source_Ortho'] = df_net['Source_AT'].map(at_to_ortho_dict)
df_net['Target_Ortho'] = df_net['Target_AT'].map(at_to_ortho_dict)
print(df_net.shape)
print(df_net.head(20))

(1709871, 4)
    Source_AT  Target_AT Source_Ortho    Target_Ortho
0   AT3G54340  AT3G09620          NaN             NaN
1   AT3G54340  AT2G42830          NaN             NaN
2   AT5G20240  AT1G69120          NaN             NaN
3   AT3G59060  AT1G03600          NaN             NaN
4   AT2G43010  AT4G04330          NaN             NaN
5   AT3G54340  AT3G02310          NaN             NaN
6   AT5G13790  AT1G27930          NaN  ORTHO04M003034
7   AT2G43010  AT2G45740          NaN             NaN
8   AT2G43010  AT3G26180          NaN             NaN
9   AT2G43010  AT5G11060          NaN  ORTHO04M021793
10  AT2G43010  AT3G59940          NaN  ORTHO04M003013
11  AT1G24260  AT5G25390          NaN             NaN
12  AT1G69120  AT4G19430          NaN             NaN
13  AT1G24260  AT4G00870          NaN  ORTHO04M014456
14  AT3G54340  AT5G53510          NaN             NaN
15  AT1G24260  AT4G38190          NaN             NaN
16  AT1G69120  AT1G75790          NaN             NaN
17  AT1G69120  

In [50]:
# Drop edges where we couldn't find a mapping and reset index
df_net_mapped = df_net.dropna(subset=['Source_Ortho', 'Target_Ortho']).reset_index(drop=True)
print(f"Successfully Mapped Edges: {len(df_net_mapped)}")
print(df_net_mapped.head())

Successfully Mapped Edges: 255035
   Source_AT  Target_AT    Source_Ortho    Target_Ortho
0  AT4G36920  AT3G61470  ORTHO04M011158  ORTHO04M010175
1  AT1G19350  AT2G38820  ORTHO04M010182  ORTHO04M008826
2  AT3G01330  AT5G62550  ORTHO04M017212  ORTHO04M021754
3  AT1G19350  AT2G20370  ORTHO04M010182  ORTHO04M012524
4  AT3G54990  AT4G37070  ORTHO04M011158  ORTHO04M014502


In [51]:
# Build PyTorch Geometric Edge Index
# We need to map the 'ORTHO...' string IDs to integer indices (0 to N-1) for PyTorch
df_expr.set_index('Gene.ID', inplace=True)
gene_to_index = {gene_id: i for i, gene_id in enumerate(df_expr.index)}
print(f"Gene to Index Sample: {list(gene_to_index.items())[:5]}")


Gene to Index Sample: [('ORTHO04M000001', 0), ('ORTHO04M000002', 1), ('ORTHO04M000003', 2), ('ORTHO04M000004', 3), ('ORTHO04M000005', 4)]


In [52]:
# Convert mapped string IDs to integer row indices
source_indices = df_net_mapped['Source_Ortho'].map(gene_to_index).values
target_indices = df_net_mapped['Target_Ortho'].map(gene_to_index).values
print(f"Sample Source Indices: {source_indices[:5]}")
print(f"Sample Target Indices: {target_indices[:5]}")


# Create the 2D edge_index tensor
edge_index = torch.tensor(np.array([source_indices, target_indices]), dtype=torch.long)
print(f"Constructed edge_index with shape: {edge_index.shape}")


Sample Source Indices: [ 9829  9297 10759  9297  9829]
Sample Target Indices: [ 9293  8355 10997 10255 10535]
Constructed edge_index with shape: torch.Size([2, 255035])


In [53]:
print("Cleaning and scaling expression features...")

# Drop the 'numeric_id' column we created for mapping so it doesn't become a feature
if 'numeric_id' in df_expr.columns:
    df_expr = df_expr.drop(columns=['numeric_id'])

# Fill missing values with 0
df_cleaned = df_expr.fillna(0.0)

# Scale the features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_cleaned)

# Convert to PyTorch Tensor
X_tensor = torch.tensor(scaled_features, dtype=torch.float32)
print(f"Feature Tensor Shape (X): {X_tensor.shape}")

Cleaning and scaling expression features...
Feature Tensor Shape (X): torch.Size([18449, 698])


In [27]:
# Define the Encoder
class ShallowEncoder(torch.nn.Module):
    def __init__(self, in_channels, out_channels):
        super(ShallowEncoder, self).__init__()
        # Single Graph Convolutional Layer
        self.conv1 = GCNConv(in_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        return x

# Initialize Model and Optimizer
in_channels = X_tensor.shape[1] # Number of expression experiments
out_channels = 16 # The size of the latent embedding space

# Instantiate the Graph Autoencoder
model = GAE(ShallowEncoder(in_channels, out_channels))

In [28]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [29]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
X_tensor = X_tensor.to(device)
edge_index = edge_index.to(device)

In [30]:
def train():
    model.train()
    optimizer.zero_grad()
    
    # Forward pass through the encoder to get latent embeddings (z)
    z = model.encode(X_tensor, edge_index)
    
    # Calculate Binary Cross Entropy loss between predicted edges and actual edges
    loss = model.recon_loss(z, edge_index)
    
    # Backpropagation
    loss.backward()
    optimizer.step()
    return float(loss)

In [31]:
print(f"Training on {device}...")
epochs = 200

for epoch in range(1, epochs + 1):
    loss = train()
    if epoch % 20 == 0:
        print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}')

Training on cuda...
Epoch: 020, Loss: 1.2591
Epoch: 040, Loss: 1.0905
Epoch: 060, Loss: 1.0436
Epoch: 080, Loss: 1.0244
Epoch: 100, Loss: 1.0135
Epoch: 120, Loss: 0.9998
Epoch: 140, Loss: 0.9887
Epoch: 160, Loss: 0.9801
Epoch: 180, Loss: 0.9763
Epoch: 200, Loss: 0.9741


In [32]:
model.eval()
with torch.no_grad():
    # z_final is your newly learned feature matrix containing structural + expression info!
    z_final = model.encode(X_tensor, edge_index).cpu().numpy()

print(f"\nTraining complete! Extracted embeddings shape: {z_final.shape}")


Training complete! Extracted embeddings shape: (18449, 16)
